<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m2_02_validation_learning_curves.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M2.02 — Too simple, too flexible, or short of data?

In the lab we drew **validation curves** (error against a setting that
makes the model more or less flexible) and **learning curves** (error
against the number of training samples). In this workbook you read both
kinds of curve **on the concrete data**, meet a model whose error
explodes, and decide whether a laboratory should pay for more tests.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "deep_tree_testing_error": "170871c6c62d5465",
    "linear_needs": "1a94ad5b45980524",
    "lowest_training_error": "e629fa6598d73276",
    "more_data_helps": "dc9c5edb8b2d479e",
    "most_flexible_k": "6b86b273ff34fce1",
    "regime_depth_1": "5a43f1e23e0f76ff",
    "regime_depth_20": "a1f32f577f69843e",
    "underfitting_k": "27badc983df1780b",
    "worst_on_unseen": "e629fa6598d73276"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data

The same laboratory tests as in workbook M2.01: seven ingredients (kg per
cubic metre), the `age` of the sample in days, and the `strength` in MPa
that we predict. Errors are mean absolute errors in MPa, measured on 30
shuffled splits.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import ShuffleSplit

concrete = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/concrete_strength.csv"
)
data = concrete.drop(columns="strength")
target = concrete["strength"]
cv = ShuffleSplit(n_splits=30, test_size=0.2, random_state=0)
concrete.head()

## 2. A validation curve: how deep should the tree be?

`max_depth` limits how many questions a decision tree may ask in a row.
A depth of 1 means a single question ("is the age below 21 days?") and
only two possible predictions. A large depth lets the tree tell almost
every sample apart.

✏️ Set `param_name` to the name of the tree parameter we want to vary
(a string).

In [ ]:
from sklearn.model_selection import ValidationCurveDisplay
from sklearn.tree import DecisionTreeRegressor

param_name = ____
depths = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20]
display = ValidationCurveDisplay.from_estimator(
    DecisionTreeRegressor(random_state=0),
    data,
    target,
    param_name=param_name,
    param_range=depths,
    cv=cv,
    scoring="neg_mean_absolute_error",
    negate_score=True,
    std_display_style="errorbar",
)
display.ax_.set(xlabel="maximum depth of the tree", ylabel="mean absolute error (MPa)")
_ = display.ax_.set_title("Validation curve of a decision tree")

The same numbers as a table:

In [ ]:
depth_curve = pd.DataFrame(
    {
        "training error": np.abs(display.train_scores).mean(axis=1),
        "testing error": np.abs(display.test_scores).mean(axis=1),
    },
    index=pd.Index(depths, name="max_depth"),
)
depth_curve.round(2)

✏️ Name the regime of the tree at each end of the curve. Write
`"underfitting"` or `"overfitting"`.

- At **depth 1**, both errors are high and almost equal.
- At **depth 20**, the training error is close to zero and far below the
  testing error.

In [ ]:
regime_depth_1 = ____
regime_depth_20 = ____
check("regime_depth_1", regime_depth_1)
check("regime_depth_20", regime_depth_20)

### ➕ Going further: the curve that does not come back up

Textbook drawings of a validation curve are U-shaped: past the best
setting, the testing error **rises** again. Look at the right-hand side of
the plot above.

✏️ Between depth 12 and depth 20, what does the testing error do? Write
`"rises sharply"`, `"levels off"` or `"falls sharply"`.

In [ ]:
deep_tree_testing_error = ____
check("deep_tree_testing_error", deep_tree_testing_error)

Overfitting is defined by the **gap** between training and testing error,
not by a testing error that climbs. A model can memorise the training set
and still be the best one you have on new data.

🧠 **Think.** In workbook M2.01 we saw that samples of one recipe sit on
both sides of a shuffled split. How could that reward a tree for
memorising? What would you change to check it?

## 3. When "more" means "simpler"

For a tree, a larger `max_depth` means a more flexible model. Not every
setting works in that direction. A k-nearest-neighbours regressor
predicts the **average strength of the k closest training samples**.

✏️ Which value of `n_neighbors` gives the **most flexible** model, the
one that follows the training data most closely: `1` or `200`?

In [ ]:
most_flexible_k = ____
check("most_flexible_k", most_flexible_k, hint="With which value does a training sample decide its own prediction alone?")

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

neighbours = [1, 2, 3, 5, 8, 12, 20, 35, 60, 100, 200]
display = ValidationCurveDisplay.from_estimator(
    make_pipeline(StandardScaler(), KNeighborsRegressor()),
    data,
    target,
    param_name="kneighborsregressor__n_neighbors",
    param_range=neighbours,
    cv=cv,
    scoring="neg_mean_absolute_error",
    negate_score=True,
    std_display_style="errorbar",
)
display.ax_.set(xlabel="number of neighbours (log scale)", ylabel="mean absolute error (MPa)")
_ = display.ax_.set_title("Validation curve of k-nearest neighbours")

This curve reads **from right to left**: the simple, rigid models are on
the right.

✏️ With which value of `n_neighbors` does the model **underfit** the most:
`1` or `200`?

In [ ]:
underfitting_k = ____
check("underfitting_k", underfitting_k)

🧠 **Think.** With `n_neighbors` equal to the number of training samples,
what does the model predict for every new sample? Which model from
workbook M2.01 is that?

## 4. ➕ Going further: an error that explodes

A small laboratory has tested only **40 samples**, all at 28 days, and
wants to predict strength from a single number: the **water/cement
ratio**. The remaining 28-day samples of our dataset play the role of the
concrete this laboratory has never seen.

In [ ]:
day28 = concrete[concrete["age"] == 28].copy()
day28["water_cement_ratio"] = day28["water"] / day28["cement"]
small_lab = day28.sample(n=40, random_state=0)
unseen = day28.drop(small_lab.index)
print(f"{len(small_lab)} samples to learn from, {len(unseen)} unseen samples")

We fit a **polynomial** of the ratio: degree 1 is a straight line, degree
4 a gentle curve, degree 15 a curve allowed to bend 14 times.

🔮 **Predict first.** Which degree will have the **lowest error on the 40
training samples**? Enter `1`, `4` or `15`.

In [ ]:
my_prediction = ____
commit(my_prediction, [1, 4, 15])

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import PolynomialFeatures


def polynomial(degree):
    return make_pipeline(StandardScaler(), PolynomialFeatures(degree), LinearRegression())


ratio_axis = pd.DataFrame(
    {"water_cement_ratio": np.linspace(small_lab["water_cement_ratio"].min(), small_lab["water_cement_ratio"].max(), 400)}
)
rows = {}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, degree in zip(axes, [1, 4, 15]):
    model = polynomial(degree).fit(small_lab[["water_cement_ratio"]], small_lab["strength"])
    rows[degree] = {
        "training error": mean_absolute_error(small_lab["strength"], model.predict(small_lab[["water_cement_ratio"]])),
        "error on unseen samples": mean_absolute_error(unseen["strength"], model.predict(unseen[["water_cement_ratio"]])),
    }
    ax.scatter(unseen["water_cement_ratio"], unseen["strength"], s=8, color="lightgray", label="unseen samples")
    ax.scatter(small_lab["water_cement_ratio"], small_lab["strength"], s=22, color="black", label="the 40 training samples")
    ax.plot(ratio_axis["water_cement_ratio"], model.predict(ratio_axis), linewidth=2)
    ax.set(ylim=(0, 90), title=f"degree {degree}", xlabel="water / cement ratio")
axes[0].set_ylabel("strength (MPa)")
axes[0].legend(loc="upper right")
degree_errors = pd.DataFrame(rows).T.rename_axis("degree")
degree_errors.round(2)

✏️ Read the table: which degree has the lowest **training** error, and
which one is by far the **worst on unseen samples**?

In [ ]:
lowest_training_error = ____
worst_on_unseen = ____
check("lowest_training_error", lowest_training_error)
check("worst_on_unseen", worst_on_unseen)

The degree-15 curve passes closest to the black dots, and between and
beyond them it shoots off to strengths no concrete has. This is the same
figure as scikit-learn's example
[Underfitting vs. overfitting](https://scikit-learn.org/1.8/auto_examples/model_selection/plot_underfitting_overfitting.html),
here with real measurements.

The validation curve of the degree shows the whole story. The laboratory
only has its 40 samples, so the curve is computed by cross-validation on
those. The vertical axis is logarithmic: each step up multiplies the
error by 10.

In [ ]:
from sklearn.model_selection import validation_curve

degrees = list(range(1, 13))
train_scores, test_scores = validation_curve(
    polynomial(1),
    small_lab[["water_cement_ratio"]],
    small_lab["strength"],
    param_name="polynomialfeatures__degree",
    param_range=degrees,
    cv=ShuffleSplit(n_splits=50, test_size=0.25, random_state=0),
    scoring="neg_mean_absolute_error",
)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(degrees, -train_scores.mean(axis=1), "o-", label="training error")
ax.plot(degrees, -test_scores.mean(axis=1), "o-", label="testing error")
ax.set(yscale="log", xlabel="degree of the polynomial", ylabel="mean absolute error (MPa)")
ax.legend()
_ = ax.set_title("Validation curve of the polynomial degree")

🧠 **Think.**

1. This curve **does** come back up, violently, while the tree's curve in
   section 2 did not. A deep tree predicts the average strength of the
   training samples that fall in a leaf. Why can such a prediction never
   be 10 000 MPa, whatever the depth?
2. The training error keeps going down as the degree grows. Why is it
   useless for choosing the degree?

## 5. A learning curve: would more tests help?

Laboratory tests are expensive: each one needs a sample, weeks of waiting
and a press. Before paying for 1000 more, we look at how the error has
changed **as the number of training samples grew**. We compare two
models:

- the fully grown decision tree (very flexible);
- a linear regression (rigid: one weight per feature).

🔮 **Predict first.** For which model is the testing error still going
down when all the data is used? **"tree"**, **"linear regression"** or
**"both"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["tree", "linear regression", "both"])

In [ ]:
from sklearn.model_selection import learning_curve

models = {
    "tree": DecisionTreeRegressor(random_state=0),
    "linear regression": LinearRegression(),
}
train_sizes = np.linspace(0.1, 1.0, 8)
curves = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, (name, model) in zip(axes, models.items()):
    sizes, train_scores, test_scores = learning_curve(
        model, data, target, train_sizes=train_sizes, cv=cv, scoring="neg_mean_absolute_error"
    )
    curves[name] = pd.DataFrame(
        {"training error": -train_scores.mean(axis=1), "testing error": -test_scores.mean(axis=1)},
        index=pd.Index(sizes, name="training samples"),
    )
    curves[name].plot(ax=ax, marker="o")
    ax.set(title=f"Learning curve of the {name}", xlabel="number of training samples")
axes[0].set_ylabel("mean absolute error (MPa)")
pd.concat(curves, axis=1).round(2)

✏️ For which model would 1000 more laboratory tests most likely pay off?
Write `"tree"` or `"linear regression"`.

In [ ]:
more_data_helps = ____
check("more_data_helps", more_data_helps)

For the linear regression, the training and testing errors have met and
stopped moving: it has learned everything a straight-line model can
learn from this data.

✏️ What does the linear regression need to get better? Write `"more
data"` or `"more flexibility"`.

In [ ]:
linear_needs = ____
check("linear_needs", linear_needs)

Let us give it some: with `PolynomialFeatures(degree=2)` the linear
regression also receives the squares of the features and their products
(cement × age, water × cement…).

In [ ]:
flexible_linear = make_pipeline(StandardScaler(), PolynomialFeatures(degree=2), LinearRegression())
sizes, train_scores, test_scores = learning_curve(
    flexible_linear, data, target, train_sizes=train_sizes, cv=cv, scoring="neg_mean_absolute_error"
)
curves["linear regression with degree 2"] = pd.DataFrame(
    {"training error": -train_scores.mean(axis=1), "testing error": -test_scores.mean(axis=1)},
    index=pd.Index(sizes, name="training samples"),
)
pd.DataFrame({name: curve["testing error"] for name, curve in curves.items()}).round(2)

🧠 **Think.**

1. Look at the first row of the last table (the smallest training set).
   Which model is the worst there? Why is a flexible model a bad idea
   when data is scarce?
2. On the tree's learning curve the training error stays near zero for
   every training set size. What does that tell you about using the
   training error to decide whether more data is needed?
3. The laboratory can afford either 1000 more tests or one week of your
   time to try other models. What do you recommend if it uses the linear
   regression? If it uses the tree?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - the validation curve of the tree: where it underfits, where it
     overfits, and why the testing error does not come back up;
   - why the k-nearest-neighbours curve reads from right to left;
   - the three polynomial fits: which one you would trust and why;
   - the two learning curves: for which model more tests are worth the
     money.